Importing the proprocessed dataset first.

In [8]:
import pandas as pd
import matplotlib.pyplot as plt

def load_dataset():
    return pd.read_csv(
        "https://raw.githubusercontent.com/emfinne/ML_Assignment1/main/AmesHousing.csv" # Replace this with the preprocessed
    )

df = load_dataset()

# Create the requested combined feature
# The dataset contains these as separate columns
df["Bsmt Full Bath + Full Bath"] = (
    df["Bsmt Full Bath"].fillna(0) +
    df["Full Bath"].fillna(0)
)

cols = [
    "MS Zoning",
    "Lot Area",
    "Utilities",
    "Bldg Type",
    "Neighborhood",
    "Overall Qual",
    "Year Remod/Add",
    "Exter Qual",
    "Bsmt Qual",
    "Total Bsmt SF",
    "Heating QC",
    "1st Flr SF",
    "2nd Flr SF",
    "Bsmt Full Bath + Full Bath",
    "Kitchen Qual",
    "Functional",
    "Garage Area",
    "Garage Qual",
    "Pool Area",
    "SalePrice"
]

# Create a smaller dataset with maybe usefull features
df_small = df[cols].copy()

Then we encode all values in the smaller dataset before splitting, so we avoid a column mismatch between the train set and test set.

In [9]:
#Array<Array<String, Dictionary>>
#NaNs are always converted to 0.
ordinalEncodedFeatures = [
    ["Garage Qual", {"Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}],
    ["Functional", {"Sal": 1, "Sev": 2, "Maj2": 3, "Maj1": 4, "Mod": 5, "Min2": 6, "Min1": 7, "Typ": 8}],
    ["Kitchen Qual", {"Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}],
    ["Heating QC", {"Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}],
    ["Bsmt Qual", {"Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}],
    ["Exter Qual", {"Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}],
    ]

#Array<String>
oneHotEncodedFeatures = [
    "Neighborhood",
    "Bldg Type",
    "Utilities",
    "MS Zoning",
    ]

def encodeData(df : pd.DataFrame, ordinalEncodedFeatures, oneHotEncodedFeatures):
    encoded = df.copy()

    #Convert feature to ordinal using the provided map
    for ordinal in ordinalEncodedFeatures:
        encoded[ordinal[0]] = encoded[ordinal[0]].map(ordinal[1]).fillna(0)

    #One hot encode feature and remove the original column
    encoded = pd.get_dummies(data=encoded, columns=oneHotEncodedFeatures)
        
    return encoded

df_small = encodeData(df_small, ordinalEncodedFeatures, oneHotEncodedFeatures)

Splitting the dataset into separate training and testing set while keeping the distributions the same.

In [10]:
from sklearn.model_selection import train_test_split

# Create price groups containing approximately equal numbers of houses
price_group = pd.qcut(
    df_small["SalePrice"],
    q=5,
    labels=False
)

train_set, test_set = train_test_split(
    df_small,
    test_size=0.30,
    random_state=42,
    stratify=price_group
)

print(train_set.shape, test_set.shape)

print("\nTraining SalePrice distribution:")
print(train_set["SalePrice"].describe())

print("\nTest SalePrice distribution:")
print(test_set["SalePrice"].describe())

(2051, 59) (879, 59)

Training SalePrice distribution:
count      2051.000000
mean     180092.995124
std       78568.123054
min       12789.000000
25%      129000.000000
50%      160000.000000
75%      212999.500000
max      755000.000000
Name: SalePrice, dtype: float64

Test SalePrice distribution:
count       879.000000
mean     182436.544937
std       82904.357479
min       55000.000000
25%      130000.000000
50%      161000.000000
75%      215500.000000
max      615000.000000
Name: SalePrice, dtype: float64


Saving the files for the separate datasets. 

In [11]:
train_set.to_csv("train_set.csv", index=False)
test_set.to_csv("test_set.csv", index=False)